<!--nav--> [🗺 Learning path](README.md) · **17/46** · ◀ [Simple MultiGPU Multimodal](./Simple_MultiGPU_Multimodal.ipynb) · [Simple MultiGPU Diffusion](./Simple_MultiGPU_Diffusion.ipynb) ▶

# Multimodal Fine-Tuning: LoRA + QLoRA + DPO

## What We're Building
A **vision-language model** that can look at images and describe them, fine-tuned 3 ways:

| Stage | Method | What it does |
|-------|--------|-------------|
| 1 | **LoRA** (8-bit model) | Teach model to describe images in our style |
| 2 | **QLoRA** (4-bit model) | Same but uses 50% less memory |
| 3 | **DPO** | Align: prefer detailed descriptions over lazy ones |
| 4 | **Merge + Export** | Combine adapters into final model |

### Model: LLaVA 1.5 7B
LLaVA = **L**arge **L**anguage **a**nd **V**ision **A**ssistant
```
Image --> CLIP Vision Encoder --> Projection --> LLaMA 7B --> Text Output
          (frozen)                (frozen)       (LoRA here)
```
We only train LoRA adapters on the language model. Vision encoder stays frozen.

### Memory Budget (T4 = 15GB)
```
8-bit model: ~7 GB  --> room for batch=4, seq=256
4-bit model: ~4 GB  --> room for batch=8, seq=256
```

---
**Runtime:** T4 GPU (Runtime > Change runtime type > T4 GPU)

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers trl datasets peft accelerate bitsandbytes pillow matplotlib

In [ ]:
import torch, gc, time

assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > T4"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU: %s (%.1f GB)" % (GPU_NAME, GPU_MEM))

def gpu_mem_gb():
    return torch.cuda.memory_allocated() / 1e9

def gpu_report(label):
    alloc = torch.cuda.memory_allocated() / 1e9
    peak = torch.cuda.max_memory_allocated() / 1e9
    print("%s | Allocated: %.2f GB | Peak: %.2f GB" % (label, alloc, peak))

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# Store results from all methods
all_results = {}

## Step 2: Load Multimodal Dataset

We use the **Naruto BLIP Captions** dataset: anime character images with text descriptions.
Small, visual, and perfect for demonstrating multimodal fine-tuning.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("lambdalabs/naruto-blip-captions", split="train")
raw_dataset = raw_dataset.shuffle(seed=42)

# Split: 200 for SFT training, 50 for DPO, 10 for testing
sft_data = raw_dataset.select(range(200))
dpo_data = raw_dataset.select(range(200, 250))
test_data = raw_dataset.select(range(250, 260))

print("SFT training: %d images" % len(sft_data))
print("DPO training: %d images" % len(dpo_data))
print("Test set: %d images" % len(test_data))
print()
print("Sample caption:", sft_data[0]["text"])

In [ ]:
import matplotlib.pyplot as plt

# Show sample images
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.imshow(sft_data[i]["image"])
    ax.set_title(sft_data[i]["text"][:40] + "...", fontsize=9)
    ax.axis("off")
plt.suptitle("Sample Training Images + Captions", fontsize=14)
plt.tight_layout()
plt.show()

## Step 3: Prepare Data for LLaVA

LLaVA uses a specific prompt format with `<image>` tokens. The processor handles converting
images to CLIP features and text to token IDs.

In [ ]:
from transformers import AutoProcessor

MODEL_NAME = "llava-hf/llava-1.5-7b-hf"
MAX_LENGTH = 768  # must be > 576 (image tokens) + text tokens

processor = AutoProcessor.from_pretrained(MODEL_NAME)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# LLaVA 1.5 prompt format
PROMPT_TEMPLATE = "USER: <image>\nDescribe this image in detail.\nASSISTANT: %s</s>"

def format_for_llava(example):
    """Add the LLaVA prompt template to each caption."""
    example["formatted_text"] = PROMPT_TEMPLATE % example["text"]
    return example

sft_dataset = sft_data.map(format_for_llava)

print("Sample formatted text:")
print(sft_dataset[0]["formatted_text"][:200])

In [ ]:
# Custom data collator for multimodal batches
# This handles both image processing (CLIP) and text tokenization together

class LLaVACollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        texts = [ex["formatted_text"] for ex in examples]
        images = [ex["image"].convert("RGB") for ex in examples]

        batch = self.processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
        )

        # Cast pixel_values to bfloat16 to match quantized model dtype
        if "pixel_values" in batch:
            batch["pixel_values"] = batch["pixel_values"].to(torch.bfloat16)

        # Labels = input_ids, mask padding tokens with -100
        labels = batch["input_ids"].clone()
        pad_id = self.processor.tokenizer.pad_token_id
        labels[labels == pad_id] = -100
        batch["labels"] = labels
        return batch

collator = LLaVACollator(processor)
print("Collator ready.")

# METHOD 1: LoRA (8-bit Model)

Load the full 7B model in **8-bit** precision (~7GB), then attach LoRA adapters
to the language model's attention layers.

```
7B model (8-bit) = ~7 GB
LoRA adapters     = ~0.1 GB
Gradients + optim = ~2 GB
Activations (b=2) = ~4 GB
Total             = ~13 GB  (fits T4!)
```

In [ ]:
from transformers import LlavaForConditionalGeneration, BitsAndBytesConfig, TrainingArguments, Trainer, TrainerCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

clear_gpu()
print("=" * 60)
print("METHOD 1: LoRA with 8-bit Model")
print("=" * 60)

# 8-bit quantization config
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
)

# Load model in 8-bit (half the size of fp16)
model_lora = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config_8bit,
    device_map="auto",
)
model_lora = prepare_model_for_kbit_training(model_lora)
gpu_report("Model loaded (8-bit)")

# LoRA on language model layers (vision encoder stays frozen)
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()
gpu_report("LoRA applied")

In [ ]:
# Track losses during training
class LossTracker(TrainerCallback):
    def __init__(self):
        self.losses = []
        self.peak_mb = 0
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.losses.append((state.global_step, logs["loss"]))
        mem = torch.cuda.max_memory_allocated() / 1e6
        if mem > self.peak_mb:
            self.peak_mb = mem

tracker_lora = LossTracker()
MAX_STEPS = 50

training_args_lora = TrainingArguments(
    output_dir="./output_lora",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=2,     # reduced: longer seqs (768) use more memory
    gradient_accumulation_steps=8,     # effective batch = 16
    learning_rate=2e-4,
    logging_steps=5,
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    save_strategy="no",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=sft_dataset,
    data_collator=collator,
    callbacks=[tracker_lora],
)

print("Training LoRA (8-bit)...")
start = time.time()
trainer_lora.train()
lora_time = time.time() - start

gpu_report("After training")
print("Time: %.1f sec" % lora_time)
print("Peak GPU: %.2f GB" % (tracker_lora.peak_mb / 1000))

all_results["lora"] = {
    "peak_gpu_gb": tracker_lora.peak_mb / 1000,
    "time_sec": lora_time,
    "losses": tracker_lora.losses,
    "batch_size": 2,
    "quant": "8-bit",
}

del model_lora, trainer_lora
clear_gpu()
print("GPU cleared.")

# METHOD 2: QLoRA (4-bit Model)

Same LoRA adapters, but the base model is **4-bit quantized** using NF4 + double quantization.

```
7B model (4-bit) = ~4 GB
LoRA adapters     = ~0.1 GB
Gradients + optim = ~2 GB
Activations (b=4) = ~5 GB
Total             = ~11 GB  (more room!)
```

In [ ]:
clear_gpu()
print("=" * 60)
print("METHOD 2: QLoRA with 4-bit Model")
print("=" * 60)

# 4-bit quantization config (NF4 + double quant = maximum compression)
# IMPORTANT: use bfloat16 compute dtype — fp16 causes grad scaler errors
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_qlora = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model_qlora = prepare_model_for_kbit_training(model_qlora)
gpu_report("Model loaded (4-bit)")

# Same LoRA config but we can target MORE layers since we have memory headroom
qlora_config = LoraConfig(
    r=64,                              # higher rank = more capacity
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=[                   # ALL linear layers in LLM
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model_qlora = get_peft_model(model_qlora, qlora_config)
model_qlora.print_trainable_parameters()
gpu_report("QLoRA applied")

In [ ]:
tracker_qlora = LossTracker()

# IMPORTANT: Use bf16=True (not fp16) for QLoRA
training_args_qlora = TrainingArguments(
    output_dir="./output_qlora",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=4,     # reduced from 8: longer seqs (768)
    gradient_accumulation_steps=4,     # effective batch = 16
    learning_rate=2e-4,
    logging_steps=5,
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    save_strategy="no",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer_qlora = Trainer(
    model=model_qlora,
    args=training_args_qlora,
    train_dataset=sft_dataset,
    data_collator=collator,
    callbacks=[tracker_qlora],
)

print("Training QLoRA (4-bit)...")
start = time.time()
trainer_qlora.train()
qlora_time = time.time() - start

gpu_report("After training")
print("Time: %.1f sec" % qlora_time)
print("Peak GPU: %.2f GB" % (tracker_qlora.peak_mb / 1000))

all_results["qlora"] = {
    "peak_gpu_gb": tracker_qlora.peak_mb / 1000,
    "time_sec": qlora_time,
    "losses": tracker_qlora.losses,
    "batch_size": 4,
    "quant": "4-bit",
}

# Keep QLoRA model for DPO (it's the most memory-efficient)
print("\nKeeping QLoRA model for DPO stage...")

## LoRA vs QLoRA Comparison

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels = ["LoRA (8-bit)", "QLoRA (4-bit)"]
colors = ["#818cf8", "#34d399"]
keys = ["lora", "qlora"]

# GPU Memory
vals = [all_results[k]["peak_gpu_gb"] for k in keys]
bars = axes[0].bar(labels, vals, color=colors, edgecolor="white", linewidth=0.5)
axes[0].axhline(y=15, color="#f59e0b", linestyle="--", label="T4 limit")
axes[0].set_title("Peak GPU Memory (GB)", fontsize=14)
axes[0].set_ylabel("GB")
axes[0].legend()
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 "%.1f" % val, ha="center", fontsize=12, fontweight="bold")

# Training Time
vals = [all_results[k]["time_sec"] for k in keys]
bars = axes[1].bar(labels, vals, color=colors, edgecolor="white", linewidth=0.5)
axes[1].set_title("Training Time (sec)", fontsize=14)
axes[1].set_ylabel("Seconds")
for bar, val in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 "%.0f" % val, ha="center", fontsize=12, fontweight="bold")

# Loss Curves
for k, label, color in zip(keys, labels, colors):
    losses = all_results[k]["losses"]
    if losses:
        axes[2].plot([x[0] for x in losses], [x[1] for x in losses],
                     color=color, linewidth=2, marker="o", markersize=4, label=label)
axes[2].set_title("Training Loss", fontsize=14)
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Loss")
axes[2].legend()
axes[2].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("\nSummary:")
print("  LoRA  (8-bit): %.1f GB peak, %d sec, batch=%d" % (
    all_results["lora"]["peak_gpu_gb"], all_results["lora"]["time_sec"], all_results["lora"]["batch_size"]))
print("  QLoRA (4-bit): %.1f GB peak, %d sec, batch=%d" % (
    all_results["qlora"]["peak_gpu_gb"], all_results["qlora"]["time_sec"], all_results["qlora"]["batch_size"]))

---
# METHOD 3: DPO Alignment (on the QLoRA model)

Now we take the QLoRA-trained model and run **DPO** to align it.

DPO teaches the model to prefer **detailed, accurate descriptions** over **vague, lazy ones**.

We create preference pairs from our image dataset:
- **Chosen**: the real detailed caption
- **Rejected**: a degraded version (shortened, generic)

```
DPO Loss = -log(sigmoid(beta * (log_ratio_chosen - log_ratio_rejected)))
```

In [ ]:
import random

# Create preference pairs: detailed caption vs degraded caption
GENERIC_CAPTIONS = [
    "a character",
    "an anime drawing",
    "a person",
    "some kind of anime character",
    "a picture",
    "a cartoon",
    "an illustration",
    "a drawing of someone",
]

def make_rejected(caption):
    """Create a bad caption: either truncate heavily or use a generic one."""
    if random.random() < 0.5:
        # Truncate to first 3 words
        words = caption.split()
        return " ".join(words[:min(3, len(words))])
    else:
        return random.choice(GENERIC_CAPTIONS)

random.seed(42)
PROMPT_TEXT = "USER: <image>\nDescribe this image in detail.\nASSISTANT:"

dpo_chosen_texts = []
dpo_rejected_texts = []
dpo_images = []

for example in dpo_data:
    chosen = example["text"]
    rejected = make_rejected(chosen)
    dpo_chosen_texts.append(PROMPT_TEXT + " " + chosen + "</s>")
    dpo_rejected_texts.append(PROMPT_TEXT + " " + rejected + "</s>")
    dpo_images.append(example["image"].convert("RGB"))

print("DPO preference pairs: %d" % len(dpo_chosen_texts))
print()
print("Example:")
print("  Chosen:   ..." + dpo_chosen_texts[0][-80:])
print("  Rejected: ..." + dpo_rejected_texts[0][-80:])

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset as TorchDataset

# Manual DPO training loop for multimodal
# This gives us full control over image+text processing

class DPOMultimodalDataset(TorchDataset):
    def __init__(self, chosen_texts, rejected_texts, images, processor):
        self.chosen_texts = chosen_texts
        self.rejected_texts = rejected_texts
        self.images = images
        self.processor = processor

    def __len__(self):
        return len(self.chosen_texts)

    def __getitem__(self, idx):
        return {
            "chosen_text": self.chosen_texts[idx],
            "rejected_text": self.rejected_texts[idx],
            "image": self.images[idx],
        }


def dpo_collate(batch, processor):
    """Process a batch for DPO: tokenize chosen and rejected with images."""
    chosen_texts = [ex["chosen_text"] for ex in batch]
    rejected_texts = [ex["rejected_text"] for ex in batch]
    images = [ex["image"] for ex in batch]

    chosen_enc = processor(
        text=chosen_texts, images=images,
        return_tensors="pt", padding=True,
    )
    rejected_enc = processor(
        text=rejected_texts, images=images,
        return_tensors="pt", padding=True,
    )

    # Cast pixel_values to bfloat16 to match QLoRA model dtype
    if "pixel_values" in chosen_enc:
        chosen_enc["pixel_values"] = chosen_enc["pixel_values"].to(torch.bfloat16)
    if "pixel_values" in rejected_enc:
        rejected_enc["pixel_values"] = rejected_enc["pixel_values"].to(torch.bfloat16)

    return chosen_enc, rejected_enc


def get_log_probs(model, inputs):
    """Get per-token log probabilities from model."""
    outputs = model(**inputs)
    logits = outputs.logits[:, :-1, :]  # shift: predict next token
    labels = inputs["input_ids"][:, 1:]  # shift: actual next tokens

    log_probs = F.log_softmax(logits.float(), dim=-1)  # compute in float32 for stability
    # Gather log probs for actual tokens
    token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)

    # Mask padding
    pad_id = processor.tokenizer.pad_token_id
    mask = (labels != pad_id).float()
    return (token_log_probs * mask).sum(-1) / mask.sum(-1).clamp(min=1)


print("DPO functions ready.")

In [ ]:
from functools import partial

# DPO needs a frozen reference model
# We use the current QLoRA model's base (before DPO updates) as reference
# To save memory, we compute reference log-probs first and cache them

print("Computing reference log-probs (frozen model)...")

dpo_dataset = DPOMultimodalDataset(dpo_chosen_texts, dpo_rejected_texts, dpo_images, processor)
dpo_loader = DataLoader(
    dpo_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=partial(dpo_collate, processor=processor),
)

# Cache reference log-probs to avoid keeping two models in memory
ref_chosen_logps = []
ref_rejected_logps = []

model_qlora.eval()
with torch.no_grad():
    for chosen_enc, rejected_enc in dpo_loader:
        chosen_enc = {k: v.to(model_qlora.device) for k, v in chosen_enc.items()}
        rejected_enc = {k: v.to(model_qlora.device) for k, v in rejected_enc.items()}
        ref_chosen_logps.append(get_log_probs(model_qlora, chosen_enc).cpu())
        ref_rejected_logps.append(get_log_probs(model_qlora, rejected_enc).cpu())

ref_chosen_logps = torch.cat(ref_chosen_logps)
ref_rejected_logps = torch.cat(ref_rejected_logps)
print("Reference log-probs cached: %d examples" % len(ref_chosen_logps))
gpu_report("After reference computation")

In [ ]:
# DPO Training Loop
BETA = 0.1        # how conservative (higher = less change)
DPO_LR = 1e-5     # lower LR for alignment
DPO_EPOCHS = 3
DPO_BATCH = 2

model_qlora.train()
optimizer = torch.optim.AdamW(
    [p for p in model_qlora.parameters() if p.requires_grad],
    lr=DPO_LR,
)

dpo_losses = []
dpo_margins = []
step = 0

print("DPO Training (beta=%.1f, lr=%s, epochs=%d)" % (BETA, DPO_LR, DPO_EPOCHS))
print("-" * 50)

for epoch in range(DPO_EPOCHS):
    epoch_loss = 0
    n_batches = 0

    dpo_loader_train = DataLoader(
        dpo_dataset,
        batch_size=DPO_BATCH,
        shuffle=False,
        collate_fn=partial(dpo_collate, processor=processor),
    )

    for batch_idx, (chosen_enc, rejected_enc) in enumerate(dpo_loader_train):
        chosen_enc = {k: v.to(model_qlora.device) for k, v in chosen_enc.items()}
        rejected_enc = {k: v.to(model_qlora.device) for k, v in rejected_enc.items()}

        # Policy log-probs
        policy_chosen_logps = get_log_probs(model_qlora, chosen_enc)
        policy_rejected_logps = get_log_probs(model_qlora, rejected_enc)

        # Reference log-probs (cached, aligned by index since shuffle=False)
        bs = policy_chosen_logps.shape[0]
        start = batch_idx * DPO_BATCH
        end = min(start + bs, len(ref_chosen_logps))
        ref_c = ref_chosen_logps[start:end].to(model_qlora.device)
        ref_r = ref_rejected_logps[start:end].to(model_qlora.device)

        # DPO loss
        chosen_rewards = BETA * (policy_chosen_logps - ref_c)
        rejected_rewards = BETA * (policy_rejected_logps - ref_r)
        margin = chosen_rewards - rejected_rewards
        loss = -F.logsigmoid(margin).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        dpo_losses.append((step, loss.item()))
        dpo_margins.append((step, margin.mean().item()))
        step += 1
        n_batches += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    avg_margin = sum(m for _, m in dpo_margins[-n_batches:]) / max(n_batches, 1)
    print("Epoch %d/%d | Loss: %.4f | Avg Margin: %.4f" % (epoch + 1, DPO_EPOCHS, avg_loss, avg_margin))

print("\nDPO training complete!")
gpu_report("After DPO")

all_results["dpo"] = {
    "losses": dpo_losses,
    "margins": dpo_margins,
}

In [ ]:
# Plot DPO training curves
plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DPO Loss
steps_l = [x[0] for x in dpo_losses]
vals_l = [x[1] for x in dpo_losses]
axes[0].plot(steps_l, vals_l, color="#818cf8", linewidth=2, marker="o", markersize=3)
axes[0].set_title("DPO Loss", fontsize=14)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.2)

# Reward Margins
steps_m = [x[0] for x in dpo_margins]
vals_m = [x[1] for x in dpo_margins]
axes[1].plot(steps_m, vals_m, color="#34d399", linewidth=2, marker="o", markersize=3)
axes[1].axhline(y=0, color="#f87171", linestyle="--", alpha=0.5, label="zero (no preference)")
axes[1].set_title("Reward Margin (chosen - rejected)", fontsize=14)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Margin")
axes[1].legend()
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("Positive margin = model prefers detailed descriptions over generic ones")

---
# Step 7: Test the Final Model on Unseen Images

In [ ]:
# Generate descriptions for test images
model_qlora.eval()

test_results = []
for i in range(min(5, len(test_data))):
    image = test_data[i]["image"].convert("RGB")
    real_caption = test_data[i]["text"]

    prompt = "USER: <image>\nDescribe this anime character in detail.\nASSISTANT:"
    inputs = processor(text=prompt, images=image, return_tensors="pt")

    # Move all tensors to device, cast pixel_values to model dtype
    inputs = {k: v.to(model_qlora.device) for k, v in inputs.items()}
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

    with torch.no_grad():
        output = model_qlora.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
        )

    generated = processor.tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    test_results.append((image, real_caption, generated))

print("Generated %d test descriptions." % len(test_results))

In [ ]:
# Show test images with generated captions
n = len(test_results)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
if n == 1:
    axes = [axes]

for i, (img, real, generated) in enumerate(test_results):
    axes[i].imshow(img)
    axes[i].axis("off")
    title = "Generated: " + generated[:60]
    if len(generated) > 60:
        title += "..."
    axes[i].set_title(title, fontsize=8, wrap=True)

plt.suptitle("Model Descriptions After QLoRA + DPO", fontsize=14)
plt.tight_layout()
plt.show()

# Print full results
for i, (img, real, generated) in enumerate(test_results):
    print("\nImage %d:" % (i + 1))
    print("  Real:      %s" % real)
    print("  Generated: %s" % generated)

---
# Step 8: Save & Export the Model

In [ ]:
# Save the QLoRA + DPO adapter
model_qlora.save_pretrained("./multimodal_qlora_dpo")
processor.save_pretrained("./multimodal_qlora_dpo")

print("Model saved to ./multimodal_qlora_dpo")
print()
print("To load later:")
print("  from peft import PeftModel")
print("  base = LlavaForConditionalGeneration.from_pretrained('%s', ...)" % MODEL_NAME)
print("  model = PeftModel.from_pretrained(base, './multimodal_qlora_dpo')")

---
# Final Summary

## What We Did

```
LLaVA 7B (vision + language model)
    |
    |-- [LoRA 8-bit]  Taught it to describe anime images (batch=4, ~13GB)
    |-- [QLoRA 4-bit] Same thing, half the memory    (batch=8, ~8GB)
    |-- [DPO]         Aligned to prefer detailed descriptions over vague ones
    |
    v
Final: QLoRA + DPO aligned multimodal model
```

## Full Training Pipeline

| Stage | Method | Purpose | GPU Used |
|-------|--------|---------|----------|
| SFT (LoRA) | 8-bit + LoRA r=32 | Learn image descriptions | ~13 GB |
| SFT (QLoRA) | 4-bit + LoRA r=64 | Same, less memory | ~8 GB |
| DPO | On QLoRA model | Prefer detailed over lazy | ~10 GB |

## What Else Could You Add?

| Technique | What it does | When to use |
|-----------|-------------|-------------|
| **ORPO** | Combines SFT + DPO in one step | Simpler alternative to SFT then DPO |
| **KTO** | Uses thumbs up/down instead of pairs | When you only have binary feedback |
| **Rejection Sampling** | Generate many, keep best | Before DPO to create better pairs |
| **Constitutional AI** | Model self-critiques | For safety alignment |
| **Merge Adapters** | Combine LoRA into base weights | For faster inference |
| **GPTQ/AWQ** | Post-training quantization | Compress final model for deployment |

### The Full Alignment Stack (what Anthropic/OpenAI do)
```
Pretrain (massive data) --> SFT (instructions) --> RLHF or DPO (preferences) --> Safety filters
```
We did steps 2 and 3 in this notebook!